<a href="https://colab.research.google.com/github/nhuvtq87/AAI2025/blob/dev/Agentic_AI_in_Customer_Service_and_Sales_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Service and Sales Chatbot
**Multi-agent chatbot using LangGraph + Google Gemini API**

Based on starter code: `code_06_XX Multi-agent chatbots with routing`

**Agents:**
- **Product Agent** — laptop features & pricing
- **Orders Agent** — order status, quantity updates & refund policy
- **Small Talk Agent** — greetings and goodbyes
- **Router Agent** — classifies intent and routes to the right agent

**Memory:** LangGraph `MemorySaver` maintains conversation state across 3+ turns

## Setup API Key


In [39]:
from google.colab import userdata
import os
os.environ["Gemini_API_Key"] = userdata.get("Gemini_API_Key")
print("API key loaded.")

API key loaded.


## Install Dependencies

In [ ]:
!pip install -q langchain-google-genai langgraph langchain langchain-community pandas
print("Done installing.")

## Cell 1 — Setup Google Gemini
Replaces `AzureChatOpenAI` from starter code

In [40]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_key = os.environ.get("Gemini_API_Key")

model = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.3,
    api_key=gemini_key
)
print("Google Gemini model ready.")

Google Gemini model ready.


## Tools
Product and order tools using CSV data

In [ ]:
import pandas as pd
from langchain_core.tools import tool

# Load laptop pricing CSV
product_pricing_df = pd.read_csv("/content/sample_data/Laptop pricing.csv")
print(product_pricing_df)

@tool
def get_laptop_price(laptop_name: str) -> str:
    """Returns the price and shipping days for a laptop given its name."""
    match_df = product_pricing_df[
        product_pricing_df["Name"].str.contains("^" + laptop_name, case=False)
    ]
    if len(match_df) == 0:
        return f"No laptop found matching '{laptop_name}'."
    return str(match_df.iloc[0].to_dict())

@tool
def get_laptop_features(laptop_name: str) -> str:
    """Returns available product info for a laptop given its name."""
    match_df = product_pricing_df[
        product_pricing_df["Name"].str.contains("^" + laptop_name, case=False)
    ]
    if len(match_df) == 0:
        return f"No laptop found matching '{laptop_name}'."
    return str(match_df.iloc[0].to_dict())

# Load orders CSV
product_orders_df = pd.read_csv("/content/sample_data/Laptop pricing.csv")
print(product_orders_df)

@tool
def get_order_details(order_id: str) -> str:
    """Returns details about a laptop order given its order ID."""
    match_df = product_orders_df[product_orders_df["Order ID"] == order_id]
    if len(match_df) == 0:
        return f"No order found with ID '{order_id}'."
    return str(match_df.iloc[0].to_dict())

@tool
def update_quantity(order_id: str, new_quantity: int) -> str:
    """Updates the quantity of laptops for a given order ID."""
    match_df = product_orders_df[product_orders_df["Order ID"] == order_id]
    if len(match_df) == 0:
        return f"No order found with ID '{order_id}'."
    product_orders_df.loc[product_orders_df["Order ID"] == order_id, "Quantity Ordered"] = new_quantity
    return f"Order {order_id} quantity updated to {new_quantity}."

@tool
def get_refund_policy(topic: str) -> str:
    """Returns the refund/return policy. Topic: general, damaged, wrong_item, or timeline."""
    policies = {
        "general": "Laptops can be returned within 30 days of delivery in original condition.",
        "damaged": "Report damaged items within 7 days with photos for a full refund or replacement.",
        "wrong_item": "Wrong items will be collected for free and replaced immediately.",
        "timeline": "Refunds are processed within 5-7 business days after item is received."
    }
    for key in policies:
        if key in topic.lower():
            return policies[key]
    return policies["general"]

print("All tools ready.")

## Base Agent Class
Reusable agent class using `StateGraph` — same pattern as `code_04`

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

class CustomerServiceAgent:

    def __init__(self, model, tools, system_prompt, debug=False):
        self.system_prompt = system_prompt
        self.debug = debug

        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("tools", self.call_tools)
        graph.add_conditional_edges("llm", self.is_tool_call, {True: "tools", False: END})
        graph.add_edge("tools", "llm")
        graph.set_entry_point("llm")

        self.memory = MemorySaver()
        self.graph = graph.compile(checkpointer=self.memory)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_llm(self, state: AgentState):
        messages = [SystemMessage(content=self.system_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        return {"messages": [result]}

    def is_tool_call(self, state: AgentState):
        return len(state["messages"][-1].tool_calls) > 0

    def call_tools(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for tc in tool_calls:
            result = self.tools[tc["name"]].invoke(tc["args"]) if tc["name"] in self.tools else "Tool not found."
            results.append(ToolMessage(tool_call_id=tc["id"], name=tc["name"], content=str(result)))
        return {"messages": results}

print("Base agent class ready.")

## Product & Orders Agents
Two specialist agents, each with their own system prompt

In [41]:
product_agent = CustomerServiceAgent(
    model=model,
    tools=[get_laptop_price, get_laptop_features],
    system_prompt="""
    You are a professional chatbot that answers questions about laptops sold by your company.
    Use the available tools to answer questions about laptop pricing and features.
    Be helpful and professional in all responses.
    """
)

orders_agent = CustomerServiceAgent(
    model=model,
    tools=[get_order_details, update_quantity, get_refund_policy],
    system_prompt="""
    You are a professional chatbot that manages laptop orders for our company.
    Use tools to retrieve order details, update quantities, and explain refund policy.
    Do NOT reveal information about other orders than the one requested.
    """
)

print("Product Agent and Orders Agent updated and ready.")

Product Agent and Orders Agent updated and ready.


## Router Agent
From `code_06` — routes each message to the correct specialist agent

In [42]:
import functools

product_node = functools.partial(agent_node, agent=product_agent, name="Product_Agent")
orders_node  = functools.partial(agent_node, agent=orders_agent,  name="Orders_Agent")

router_agent = RouterAgent(
    model=model,
    system_prompt="""
You are a Router. Analyze the user's message and respond with exactly one word:
SMALLTALK  - greetings, goodbyes, or casual chat
PRODUCT    - questions about laptop features, specs, or pricing
ORDER      - questions about orders, delivery, quantity, or refund policy
END        - anything else

Respond with ONLY that one word. Nothing else.
""",
    smalltalk_prompt="""
Respond professionally to greetings and goodbyes.
Let the user know you can help with laptop product questions and order management.
""",
    debug=False
)

print("Router Agent updated. Full multi-agent chatbot system is now online and verified!")

Router Agent updated. Full multi-agent chatbot system is now online and verified!


## Multi-Turn Conversation Demo
Shows all 3 agent types with MemorySaver keeping context

In [ ]:
import uuid
from langchain_core.messages import HumanMessage

user_inputs = [
    "Hello, how are you?",
    "Tell me about the SpectraBook S",
    "How much does it cost?",
    "Show me the details of order ORD-7311",
    "Can you add one more laptop to that order?",
    "What is the general refund policy for laptops?",
    "Compare GammaAir X and AlphaBook Pro prices",
    "Goodbye!",
]

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("\n" + "="*60)
print("   CUSTOMER SERVICE CHATBOT — MULTI-AGENT DEMO")
print("="*60)

for user_input in user_inputs:
    print(f"\n{'─'*55}")
    print(f"USER  : {user_input}")
    try:
        response = router_agent.graph.invoke(
            {"messages": [HumanMessage(user_input)]}, config=config
        )
        print(f"AGENT : {response['messages'][-1].content}")
    except Exception as e:
        print(f"AGENT error executing model call: {e}")

print("\n" + "="*60)
print("DEMO COMPLETE")
print("="*60)

## Cell 7 — Memory Test: 3-Turn Conversation
Shows MemorySaver retaining context across 3 turns

In [43]:
import uuid
from langchain_core.messages import HumanMessage

print("="*55)
print("MEMORY TEST — 3-Turn Conversation")
print("="*55)

memory_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

turns = [
    "What are the details of order ID ORD-6948?",
    "Can you increase the quantity of it to 5?",
    "Can you show me the updated details of the order?",
]

for turn in turns:
    print(f"\n{'─'*50}")
    print(f"USER  : {turn}")
    try:
        result = router_agent.graph.invoke(
            {"messages": [HumanMessage(turn)]}, config=memory_config
        )
        print(f"AGENT : {result['messages'][-1].content}")
    except Exception as e:
        print(f"AGENT error executing model call: {e}")
        print("\n[TIP] Please change model='gemini-1.5-flash' to model='models/gemini-1.5-flash' in your model setup cell, re-execute the setup, and run this cell again.")
        break

print("\nMemorySaver execution process completed.")

MEMORY TEST — 3-Turn Conversation

──────────────────────────────────────────────────
USER  : What are the details of order ID ORD-6948?
AGENT error executing model call: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

[TIP] Please change model='gemini-1.5-flash' to model='models/gemini-1.5-flash' in your model setup cell, re-execute the setup, and run this cell again.

MemorySaver execution process completed.
